# Vocational NMT: run 6, terminology-aware distillation on case-fixed drafts

What changed since run 5: Maithili and Konkani case endings that placeholder repair had
detached ("विद्युतरोधन क") are re-attached in the repaired training drafts (1,050 mai and 427 gom rows).
The repaired drafts ship in the bundle, so this notebook does no translation of training data.

**GPU 0:** `joint_formal_v2` (seed 42), the main run 6 model → domain evaluation (plain and hybrid) → IN22-Gen forgetting check.
**GPU 1, in parallel:** `joint_formal_v2_s43` (seed 43, for seed variance) → `hi_formal_v2` (Hindi only, for the joint-vs-single ablation) → their evaluations.

**Input:** only the latest bundle dataset. Use GPU T4 x2, Internet on, and the `HF_TOKEN` secret. Then **Save & Run All** (about 3 hours); the outputs end up in `/kaggle/working/outputs.zip`.

In [ ]:
# 1. Copy the newest stamped bundle into a writable working directory (+ reuse earlier drafts)
import os, shutil, subprocess, sys, glob, zipfile
from pathlib import Path

PROJECT = Path("/kaggle/working/project")
# Only folders/zips carrying kaggle/BUNDLE_VERSION count as code; outputs of earlier runs
# (added as inputs to reuse drafts) also contain a backend/ folder with OLD code.
candidates = []
for z in glob.glob("/kaggle/input/**/vocational-nmt-bundle.zip", recursive=True):
    with zipfile.ZipFile(z) as zf:
        if "kaggle/BUNDLE_VERSION" in zf.namelist():
            candidates.append((zf.read("kaggle/BUNDLE_VERSION").decode().strip(), z))
for stamp in glob.glob("/kaggle/input/**/kaggle/BUNDLE_VERSION", recursive=True):
    candidates.append((Path(stamp).read_text().strip(), str(Path(stamp).parents[1])))
if not candidates:
    raise FileNotFoundError("No bundle with kaggle/BUNDLE_VERSION under /kaggle/input - upload the latest bundle zip.")
print("bundles attached:", sorted({v for v, _ in candidates}))
version, source = max(candidates)

# /kaggle/working survives while an interactive session is open: re-copy when the attached
# bundle is newer than the copy from an earlier cell run.
copied = PROJECT / "kaggle/BUNDLE_VERSION"
if PROJECT.exists() and (not copied.exists() or copied.read_text().strip() != version):
    print("Replacing stale project copy", copied.read_text().strip() if copied.exists() else "(unstamped)")
    shutil.rmtree(PROJECT)
if not PROJECT.exists():
    print(f"Using bundle {version} from {source}")
    if source.endswith(".zip"):
        zipfile.ZipFile(source).extractall(PROJECT)
    else:
        shutil.copytree(source, PROJECT)
print("bundle version:", copied.read_text().strip())
os.chdir(PROJECT)
os.environ["PYTHONPATH"] = str(PROJECT)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation during beam search
sys.path.insert(0, str(PROJECT))

CORPUS = PROJECT / "backend/data/corpus_v2"
for pattern, sub in (("drafts_*.csv", ""), ("postedit/test_*.csv", "postedit")):
    for src in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True):
        dst = CORPUS / sub / Path(src).name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(src, dst)
            print("reused", src)
print(sorted(p.name for p in CORPUS.iterdir()))
assert (PROJECT / "backend/models/it2_lora_joint/adapter_model.safetensors").exists(), "run-4 adapter missing from bundle"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install the pinned stack (about 1 minute)
!pip install -q -r kaggle/requirements-kaggle.txt
!python -c "import transformers, peft; print('transformers', transformers.__version__, '| peft', peft.__version__)"

In [ ]:
# 3. Hugging Face token + access check for the gated repos
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import hf_hub_download
IN22_OK = True
for repo, kind, fname in [("ai4bharat/indictrans2-en-indic-dist-200M", "model", "config.json"),
                          ("ai4bharat/indictrans2-en-indic-1B", "model", "config.json"),
                          ("ai4bharat/IN22-Gen", "dataset", "data/train-00000-of-00001.parquet")]:
    try:
        hf_hub_download(repo, fname, repo_type=kind, token=os.environ["HF_TOKEN"])
        print("access OK:", repo)
    except Exception as exc:
        print(f"NO ACCESS: {repo} -> accept its terms on huggingface.co ({type(exc).__name__})")
        if kind == "model":
            raise
        IN22_OK = False

In [ ]:
# 4. Checks + smoke test (~5 min): case-fixed formal drafts present, 10 training steps, adapter loading
def check(step):
    assert _exit_code == 0, f"{step} failed - see the output above"

import re
import pandas as pd
for s in ["hi", "gom", "mai", "doi"]:
    assert (CORPUS / f"drafts_{s}_formal.csv").exists(), f"drafts_{s}_formal.csv missing from the bundle"
mai = pd.read_csv(CORPUS / "drafts_mai_formal.csv", keep_default_na=False)
detached = mai[mai.term_repair == "repaired"].target_sentence.str.contains(r"(?:न|ला|क्ष|र्क) (?:क|केँ|सँ|मे)(?=[\s।.,])").sum()
print("Maithili repaired drafts with a detached case ending after a term-like word:", detached, "(run 5 data: >1000)")

!CUDA_VISIBLE_DEVICES=0 python -m backend.finetune --drafts-variant formal --run-name smoke --max-steps 10 --eval-steps 5 --dev-per-lang 8
check("training smoke test")
!python -m backend.baseline_inference --model backend/models/it2_lora_smoke --text "Test the insulation with a megger." --target-lang mai_Deva
check("adapter loading")
shutil.rmtree("backend/models/it2_lora_smoke", ignore_errors=True)

## Step 1: training (GPU 0: main model; GPU 1: seed 43, then Hindi-only)

In [ ]:
import torch
n_gpu = torch.cuda.device_count()
gpu1 = None
if n_gpu >= 2:
    chain = ("python -m backend.finetune --drafts-variant formal --run-name joint_formal_v2_s43 --seed 43 && "
             "python -m backend.finetune --drafts-variant formal --run-name hi_formal_v2 --langs hin_Deva")
    gpu1 = subprocess.Popen(["bash", "-c", chain], env=dict(os.environ, CUDA_VISIBLE_DEVICES="1"),
                            stdout=open("/kaggle/working/train_gpu1.log", "w"), stderr=subprocess.STDOUT)
else:
    print("Only one GPU: the seed-43 and Hindi-only runs are skipped.")

!CUDA_VISIBLE_DEVICES=0 python -m backend.finetune --drafts-variant formal --run-name joint_formal_v2
check("main fine-tuning")

## Step 2: evaluation

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m backend.run_domain_eval --systems lora_joint_formal_v2,lora_joint_formal_v2+hy --tag run6a
check("evaluation of the main model")

if gpu1 is not None:
    print("GPU 1 training exit code:", gpu1.wait())
    print(open("/kaggle/working/train_gpu1.log").read()[-2500:])
    gpu1_eval = subprocess.Popen(
        ["bash", "-c",
         "python -m backend.run_domain_eval --systems lora_joint_formal_v2_s43,lora_joint_formal_v2_s43+hy --tag run6b && "
         "python -m backend.run_domain_eval --langs hin_Deva --systems lora_hi_formal_v2,lora_hi_formal_v2+hy --tag run6c"],
        env=dict(os.environ, CUDA_VISIBLE_DEVICES="1"),
        stdout=open("/kaggle/working/eval_gpu1.log", "w"), stderr=subprocess.STDOUT)
else:
    gpu1_eval = None

if IN22_OK:
    !CUDA_VISIBLE_DEVICES=0 python -m backend.run_baseline_eval --skip-base --adapter backend/models/it2_lora_joint_formal_v2 --output backend/results/in22_gen_lora_formal_v2.csv
else:
    print("Skipped IN22-Gen: no access.")

if gpu1_eval is not None:
    print("GPU 1 evaluation exit code:", gpu1_eval.wait())
    print(open("/kaggle/working/eval_gpu1.log").read()[-3000:])

In [ ]:
# Package everything that must come back to your machine
out = Path("/kaggle/working/outputs")
shutil.rmtree(out, ignore_errors=True)
out.mkdir(parents=True)
for run in ["joint_formal_v2", "joint_formal_v2_s43", "hi_formal_v2"]:
    src = Path(f"backend/models/it2_lora_{run}")
    if src.exists():
        shutil.copytree(src, out / f"it2_lora_{run}", ignore=shutil.ignore_patterns("checkpoints"))
    else:
        print("missing:", src)
shutil.copytree("backend/results", out / "results")
for log in glob.glob("/kaggle/working/*.log"):
    shutil.copy(log, out)
shutil.make_archive("/kaggle/working/outputs", "zip", out)
!du -sh /kaggle/working/outputs.zip